# Genre-based content recommendations

This notebook tests whether movie genres provide useful recommendation signals beyond the popularity baseline. Each movie is represented by its genre membership, and each user's recommendations are based on their positive training history.

Run notebook 01 first. We load its canonical **per-user chronological 60/20/20** partitions, use the same training candidate catalog as collaborative filtering, and reserve test feedback for the final evaluation. Validation users, the positive-rating threshold, Recall@10 denominators, exclusions, and tie rules remain unchanged across models.

## Load the canonical split and evaluation contract

The saved manifest and file fingerprints make the data dependency explicit. Loading training and validation files from the split-specific output directory avoids independently reconstructing partitions or reusing artifacts from the former global date split. Test feedback is not loaded here.

In [1]:
from pathlib import Path
import hashlib
import json

import pandas as pd

SPLIT_ID = "per_user_chronological_60_20_20_v1"
POSITIVE_THRESHOLD = 4.0
TOP_K = 10

# Resolve the project from either its root or any nested notebook directory.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "raw" / "ml-latest-small").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run from the project containing data/raw/ml-latest-small.")

raw_dir = data_dir = project_root / "data" / "raw" / "ml-latest-small"
processed_dir = project_root / "data" / "processed" / SPLIT_ID
manifest_path = processed_dir / "split_manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("Run notebook 01 first to write the canonical split.")
split_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert split_manifest["split_id"] == SPLIT_ID
assert split_manifest["positive_threshold"] == POSITIVE_THRESHOLD
assert split_manifest["top_k"] == TOP_K

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename, expected_hash in split_manifest["raw_sha256"].items():
    if file_sha256(raw_dir / filename) != expected_hash:
        raise ValueError(f"{filename} changed; regenerate the split in notebook 01.")

movies = pd.read_csv(raw_dir / "movies.csv")
loaded_splits = {}
for name in ("train", "validation"):
    path = processed_dir / f"{name}_ratings.csv"
    expected = split_manifest["files"][path.name]
    if file_sha256(path) != expected["sha256"]:
        raise ValueError(f"{path.name} does not match the split manifest.")
    loaded_splits[name] = pd.read_csv(path, parse_dates=["rated_at"])
    assert len(loaded_splits[name]) == expected["rows"]

train_ratings = loaded_splits["train"]
validation_ratings = loaded_splits["validation"]

def save_validation_results(results, filename):
    """Attach the split and evaluation contract to each per-user result row."""
    saved = results.copy()
    saved["split_id"] = SPLIT_ID
    saved["split_fingerprint"] = split_manifest["split_fingerprint"]
    saved["evaluation_split"] = "validation"
    saved["positive_threshold"] = POSITIVE_THRESHOLD
    saved["k"] = TOP_K
    saved.to_csv(processed_dir / filename, index=False)

def validate_result_provenance(results, label="results"):
    expected = {
        "split_id": SPLIT_ID,
        "split_fingerprint": split_manifest["split_fingerprint"],
        "evaluation_split": "validation",
        "positive_threshold": POSITIVE_THRESHOLD,
        "k": TOP_K,
    }
    for column, value in expected.items():
        assert column in results and results[column].eq(value).all(), (
            f"{label}: missing or incompatible {column}; rerun its source notebook."
        )

print("Split:", SPLIT_ID)
print("Training ratings:", len(train_ratings))
print("Validation ratings:", len(validation_ratings))


Split: per_user_chronological_60_20_20_v1
Training ratings: 60255
Validation ratings: 20164


## Restrict the feature catalog to training candidates

Although genres are available for the full MovieLens catalog, this benchmark ranks only movies represented in training. The restriction keeps candidate coverage comparable with collaborative filtering. The complete held-out positive set still defines each user's recall denominator.

In [2]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

content_movies = (
    movies.loc[
        movies["movieId"].isin(train_ratings["movieId"]),
        ["movieId", "title", "genres"],
    ]
    .sort_values("movieId")
    .reset_index(drop=True)
    .copy()
)
assert content_movies["movieId"].is_unique
print("Training candidate movies:", len(content_movies))
content_movies.head()


Training candidate movies: 6885


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## Parse genre membership

MovieLens stores multiple genre labels in a pipe-delimited string. We convert it to a list before multi-hot encoding. Missing labels and the explicit no-genre marker become an empty list, preserving the candidate with a zero feature vector instead of dropping it.

In [3]:
def split_genres(genre_text):
    """Convert a MovieLens genre string to a list of feature labels."""
    if pd.isna(genre_text) or genre_text == "(no genres listed)":
        return []
    return genre_text.split("|")

content_movies["genre_list"] = content_movies["genres"].apply(split_genres)
content_movies[["movieId", "title", "genre_list"]].head()


,movieId,title,genre_list
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


## Build the multi-hot feature matrix

Each row corresponds to a movie ID and each column to a genre observed in the training catalog. Values indicate membership. Retaining IDs as the index ensures similarity arrays can be mapped back to the correct catalog rows. Movies with identical genre vectors are indistinguishable to this representation.

In [4]:
from sklearn.preprocessing import MultiLabelBinarizer

genre_encoder = MultiLabelBinarizer()
genre_values = genre_encoder.fit_transform(content_movies["genre_list"])
genre_matrix = pd.DataFrame(
    genre_values,
    index=content_movies["movieId"],
    columns=genre_encoder.classes_,
)
assert len(genre_encoder.classes_) > 0, "The training catalog has no usable genre labels."
assert set(genre_matrix.index) == set(train_ratings["movieId"])
print("Genre matrix shape:", genre_matrix.shape)
genre_matrix.iloc[:5, :8]


Genre matrix shape: (6885, 19)


,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama
movieId,,,,,,,,
1,0,1,1,1,1,0,0,0
2,0,1,0,1,0,0,0,0
3,0,0,0,0,1,0,0,0
4,0,0,0,0,1,0,0,1
5,0,0,0,0,1,0,0,0


## Inspect a pairwise genre similarity

Cosine similarity measures how much genre membership overlaps after accounting for the number of genres attached to each title. A movie with no listed genres has a zero vector and contributes zero similarity. The example uses Toy Story and Jumanji when present.

In [5]:
seed_a = 1 if 1 in genre_matrix.index else int(genre_matrix.index[0])
seed_b = 2 if 2 in genre_matrix.index else int(genre_matrix.index[-1])

genre_similarity = cosine_similarity(genre_matrix.loc[[seed_a]], genre_matrix.loc[[seed_b]])[0, 0]
print("Movie IDs:", seed_a, seed_b)
print("Genre counts:", int(genre_matrix.loc[seed_a].sum()), int(genre_matrix.loc[seed_b].sum()))
print("Genre cosine similarity:", genre_similarity)


Movie IDs: 1 2
Genre counts: 5 3
Genre cosine similarity: 0.7745966692414834


## Build training popularity for tie-breaking and fallback

Genre features frequently produce tied scores, so training rating counts provide a stable secondary ordering. They also give users without positive history a complete popularity fallback. These counts use training interactions only.

In [6]:
movie_popularity = (
    train_ratings.groupby("movieId")
    .size()
    .reset_index(name="training_rating_count")
    .merge(movies[["movieId", "title"]], on="movieId", how="left", validate="one_to_one")
    .sort_values(["training_rating_count", "movieId"], ascending=[False, True])
    .reset_index(drop=True)
)

assert set(movie_popularity["movieId"]) == set(train_ratings["movieId"])
movie_popularity.head(TOP_K)


,movieId,training_rating_count,title
0,356,257,Forrest Gump (1994)
1,318,251,"Shawshank Redemption, The (1994)"
2,296,240,Pulp Fiction (1994)
3,2571,234,"Matrix, The (1999)"
4,593,220,"Silence of the Lambs, The (1991)"
5,260,206,Star Wars: Episode IV - A New Hope (1977)
6,110,196,Braveheart (1995)
7,480,183,Jurassic Park (1993)
8,589,171,Terminator 2: Judgment Day (1991)
9,1,168,Toy Story (1995)


## Inspect genre neighbors of a seed movie

The neighbor list makes the feature representation inspectable before using it for personalization. A seed can have many equally similar neighbors because genre labels are coarse; popularity and movie ID resolve those ties deterministically.

In [7]:
seed_scores = cosine_similarity(genre_matrix.loc[[seed_a]], genre_matrix)[0]
similar_genre_movies = (
    pd.DataFrame({"movieId": genre_matrix.index, "similarity": seed_scores})
    .merge(content_movies[["movieId", "title", "genres"]], on="movieId", validate="one_to_one")
    .merge(movie_popularity[["movieId", "training_rating_count"]], on="movieId", validate="one_to_one")
)
similar_genre_movies = (
    similar_genre_movies.loc[similar_genre_movies["movieId"] != seed_a]
    .sort_values(["similarity", "training_rating_count", "movieId"], ascending=[False, False, True])
    .reset_index(drop=True)
)
similar_genre_movies.head(TOP_K)


,movieId,similarity,title,genres,training_rating_count
0,4886,1.0,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,95
1,3114,1.0,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,77
2,2294,1.0,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,29
3,4016,1.0,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,25
4,53121,1.0,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,9
5,3754,1.0,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,7
6,166461,1.0,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,6
7,45074,1.0,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1
8,65577,1.0,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,1
9,136016,1.0,The Good Dinosaur (2015),Adventure|Animation|Children|Comedy|Fantasy,1


## Score candidates from one user's training likes

Each candidate receives its mean genre similarity to the movies the user rated at least 4.0 in training. The example uses zero scores when there are no positive ratings, preserving the same popularity fallback used in the general function.

In [8]:
example_user_id = 1 if 1 in set(train_ratings["userId"]) else int(train_ratings["userId"].iloc[0])
user_history = train_ratings.loc[train_ratings["userId"] == example_user_id]
liked_ids = user_history.loc[user_history["rating"] >= POSITIVE_THRESHOLD, "movieId"]

if liked_ids.empty:
    mean_scores = np.zeros(len(genre_matrix), dtype=float)
else:
    mean_scores = cosine_similarity(genre_matrix.loc[liked_ids], genre_matrix).mean(axis=0)

user_genre_scores = pd.Series(mean_scores, index=genre_matrix.index, name="score")
print("Example user:", example_user_id)
print("Training ratings:", len(user_history))
print("Positive training ratings:", len(liked_ids))


Example user: 1
Training ratings: 139
Positive training ratings: 121


## Exclude rated movies and rank the example

All previously rated training titles are removed, regardless of their rating. Candidate order uses descending genre score, descending training popularity, and ascending movie ID. The score summarizes feature similarity; it is not a predicted rating or a probability of liking the movie.

In [9]:
genre_recommendations = movie_popularity.loc[
    ~movie_popularity["movieId"].isin(user_history["movieId"])
].copy()
genre_recommendations["score"] = genre_recommendations["movieId"].map(user_genre_scores).fillna(0.0)
genre_recommendations = (
    genre_recommendations.sort_values(
        ["score", "training_rating_count", "movieId"], ascending=[False, False, True]
    )
    .head(TOP_K)
    .reset_index(drop=True)
)
genre_recommendations


,movieId,training_rating_count,title,score
0,117646,1,Dragonheart 2: A New Beginning (2000),0.501345
1,55116,2,"Hunting Party, The (2007)",0.493151
2,4956,1,"Stunt Man, The (1980)",0.481357
3,2683,84,Austin Powers: The Spy Who Shagged Me (1999),0.479557
4,6156,17,Shanghai Knights (2003),0.479557
5,2471,16,Crocodile Dundee II (1988),0.479557
6,6764,10,"Rundown, The (2003)",0.479557
7,138036,9,The Man from U.N.C.L.E. (2015),0.479557
8,34520,6,"Dukes of Hazzard, The (2005)",0.479557
9,2748,5,Allan Quatermain and the Lost City of Gold (1987),0.479557


## Define the reusable genre recommender

This function implements the walkthrough for any user without changing the candidate set. Empty positive histories and zero genre similarity leave all scores at zero, causing the shared popularity ordering to supply the fallback.

In [10]:
def recommend_genre_based(
    user_id,
    training_data,
    genre_matrix,
    popularity_ranking,
    n=10,
):
    """Rank unseen fitting-catalog movies by mean genre similarity to training likes."""
    history = training_data.loc[training_data["userId"] == user_id]
    liked_ids = history.loc[
        (history["rating"] >= POSITIVE_THRESHOLD)
        & history["movieId"].isin(genre_matrix.index),
        "movieId",
    ]

    recommendations = popularity_ranking.loc[
        ~popularity_ranking["movieId"].isin(history["movieId"])
    ].copy()
    recommendations["score"] = 0.0

    if not liked_ids.empty:
        similarities = cosine_similarity(genre_matrix.loc[liked_ids], genre_matrix)
        scores = pd.Series(similarities.mean(axis=0), index=genre_matrix.index)
        recommendations["score"] = recommendations["movieId"].map(scores).fillna(0.0)

    return (
        recommendations.sort_values(
            ["score", "training_rating_count", "movieId"],
            ascending=[False, False, True],
        )
        .head(n)
        .reset_index(drop=True)
    )


## Verify the walkthrough against the reusable function

Agreement on the example's ranked IDs catches inconsistencies between exploratory cells and the function used during evaluation. Additional examples make the output easy to inspect without relying on a fixed result table.

In [11]:
function_example = recommend_genre_based(
    example_user_id, train_ratings, genre_matrix, movie_popularity, n=TOP_K
)
assert function_example["movieId"].tolist() == genre_recommendations["movieId"].tolist()

for user_id in sorted(train_ratings["userId"].unique())[:2]:
    print("Recommendations for user:", user_id)
    display(recommend_genre_based(user_id, train_ratings, genre_matrix, movie_popularity, n=TOP_K))


Recommendations for user: 1


,movieId,training_rating_count,title,score
0,117646,1,Dragonheart 2: A New Beginning (2000),0.501345
1,55116,2,"Hunting Party, The (2007)",0.493151
2,4956,1,"Stunt Man, The (1980)",0.481357
3,2683,84,Austin Powers: The Spy Who Shagged Me (1999),0.479557
4,6156,17,Shanghai Knights (2003),0.479557
5,2471,16,Crocodile Dundee II (1988),0.479557
6,6764,10,"Rundown, The (2003)",0.479557
7,138036,9,The Man from U.N.C.L.E. (2015),0.479557
8,34520,6,"Dukes of Hazzard, The (2005)",0.479557
9,2748,5,Allan Quatermain and the Lost City of Gold (1987),0.479557


Recommendations for user: 2


,movieId,training_rating_count,title,score
0,145,35,Bad Boys (1995),0.531666
1,20,12,Money Train (1995),0.531666
2,7007,5,"Last Boy Scout, The (1991)",0.531666
3,1432,4,Metro (1997),0.531666
4,5628,3,Wasabi (2001),0.531666
5,5027,2,Another 48 Hrs. (1990),0.531666
6,2000,46,Lethal Weapon (1987),0.509273
7,2001,34,Lethal Weapon 2 (1989),0.509273
8,4085,29,Beverly Hills Cop (1984),0.509273
9,2002,28,Lethal Weapon 3 (1992),0.509273


## Evaluate the common validation cohort

Recall@10 is computed for every user with positive validation feedback and then averaged with equal weight per user. A user with no positive training ratings stays in the cohort and receives fallback recommendations. A user with no positive validation rating has an undefined denominator and is reported separately.

Positive validation movies outside the training catalog remain in the denominator. The loop records catalog coverage and checks recommendation uniqueness, list length, candidate membership, rated-movie exclusions, and finite scores.

In [12]:
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= POSITIVE_THRESHOLD
].copy()

training_users = set(train_ratings["userId"])
liked_training_users = set(
    train_ratings.loc[train_ratings["rating"] >= POSITIVE_THRESHOLD, "userId"]
)
candidate_ids = set(movie_popularity["movieId"])
evaluation_rows = []

for user_id, positives in validation_positives.groupby("userId", sort=True):
    positive_ids = set(positives["movieId"])
    seen_ids = set(train_ratings.loc[train_ratings["userId"] == user_id, "movieId"])
    recommendations = recommend_genre_based(
        user_id=user_id,
        training_data=train_ratings,
        genre_matrix=genre_matrix,
        popularity_ranking=movie_popularity,
        n=TOP_K,
    )
    recommended_ids = set(recommendations["movieId"])

    assert recommendations["movieId"].is_unique
    assert len(recommendations) == min(TOP_K, len(candidate_ids - seen_ids))
    assert recommended_ids.issubset(candidate_ids)
    assert recommended_ids.isdisjoint(seen_ids)
    assert np.isfinite(recommendations["score"]).all()

    hits = len(positive_ids & recommended_ids)
    evaluation_rows.append({
        "userId": user_id,
        "positive_movie_count": len(positive_ids),
        "positive_movies_in_training_catalog": len(positive_ids & candidate_ids),
        "recommended_movie_count": len(recommendations),
        "hits_at_10": hits,
        "recall_at_10": hits / len(positive_ids),
        "has_training_history": user_id in training_users,
        "has_liked_training_history": user_id in liked_training_users,
    })

genre_based_results = pd.DataFrame(evaluation_rows)
assert not genre_based_results.empty, "No users have positive validation ratings."
assert genre_based_results["userId"].is_unique
assert genre_based_results["has_training_history"].all()

print("Users evaluated:", len(genre_based_results))
print("Users with no positive validation ratings:",
      validation_ratings["userId"].nunique() - len(genre_based_results))
print("Mean Recall@10:", genre_based_results["recall_at_10"].mean())
genre_based_results.head()


Users evaluated: 594
Users with no positive validation ratings: 16
Mean Recall@10: 0.008459947958151061


,userId,positive_movie_count,positive_movies_in_training_catalog,recommended_movie_count,hits_at_10,recall_at_10,has_training_history,has_liked_training_history
0,1,40,40,10,0,0.0,True,True
1,2,5,5,10,0,0.0,True,True
2,3,5,3,10,0,0.0,True,True
3,4,19,17,10,0,0.0,True,True
4,5,5,5,10,0,0.0,True,True


## Check how much positive training history is available

Every user has training history under this split, but not everyone necessarily has a liked training movie. This summary distinguishes users with a personalized signal from users who rely on fallback recommendations.

In [13]:
personalization_summary = (
    genre_based_results.groupby("has_liked_training_history")
    .agg(users=("userId", "size"), mean_recall_at_10=("recall_at_10", "mean"),
         total_hits_at_10=("hits_at_10", "sum"))
    .reset_index()
)
personalization_summary


,has_liked_training_history,users,mean_recall_at_10,total_hits_at_10
0,False,1,0.000000,0
1,True,593,0.008474,67


## Compare with the same users' popularity results

The baseline file must match the current split fingerprint. An outer join checks that both models evaluated exactly the same users, and the denominator check verifies the same relevant sets were counted. Paired recall changes make user-level gains and losses visible alongside the aggregate average.

In [14]:
popularity_results = pd.read_csv(processed_dir / "popularity_validation_results.csv")
validate_result_provenance(popularity_results)

paired_results = genre_based_results.merge(
    popularity_results[["userId", "positive_movie_count", "recall_at_10"]],
    on="userId",
    how="outer",
    suffixes=("_model", "_popularity"),
    indicator=True,
    validate="one_to_one",
)
assert paired_results["_merge"].eq("both").all()
assert (
    paired_results["positive_movie_count_model"]
    == paired_results["positive_movie_count_popularity"]
).all()

paired_results["recall_change"] = (
    paired_results["recall_at_10_model"] - paired_results["recall_at_10_popularity"]
)

validation_comparison = pd.DataFrame({
    "model": ["Popularity", "Genres"],
    "users": [len(paired_results), len(paired_results)],
    "mean_recall_at_10": [
        paired_results["recall_at_10_popularity"].mean(),
        paired_results["recall_at_10_model"].mean(),
    ],
})
display(validation_comparison)

change = paired_results["recall_change"]
print("Users with improved recall:", change.gt(0).sum())
print("Users with worse recall:", change.lt(0).sum())
print("Users with unchanged recall:", change.eq(0).sum())
print("Mean gain in percentage points:", 100 * change.mean())


,model,users,mean_recall_at_10
0,Popularity,594,0.072092
1,Genres,594,0.008460


Users with improved recall: 41
Users with worse recall: 203
Users with unchanged recall: 350
Mean gain in percentage points: -6.36318437582146


## Save validation results with provenance

Notebook 06 reads this file alongside the other model evaluations. The shared writer attaches the split identity, fingerprint, recommendation-list size, and positive threshold so downstream comparisons can detect incompatible artifacts.

In [15]:
save_validation_results(genre_based_results, "genre_validation_results.csv")
print("Saved genre validation results.")


Saved genre validation results.


## Interpretation and limitations

Use the freshly computed metrics to assess whether genre similarity improves retrieval on this split. Prior scores from the global calendar split have been removed because their user coverage and evaluation conditions differ.

Genres provide a deliberately coarse content baseline. Identical genre vectors cannot distinguish different films, averaging likes can dilute a user's varied interests, and zero-feature movies rely on popularity tie-breaking. The shared training candidate catalog also limits which held-out positives are retrievable.

Per-user chronology preserves an individual's rating order but does not reproduce one global deployment date. This is an existing-user offline benchmark. The test partition remains reserved until model choice and settings have been fixed using validation.